# CNN Training Notebook

### 0. Import Library & Dataset

In [4]:
# Standard libraries
import os
import sys
import itertools
import time
import json
import glob

from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualisasi
import matplotlib.pyplot as plt

# Evaluasi Model
from sklearn.metrics import f1_score, classification_report

# Deep Learning
import tensorflow as tf

print('TF: ', tf.__version__)
print('GPU: ', tf.config.list_physical_devices('GPU'))

TF:  2.10.0
GPU:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
src = Path(os.getcwd()) # Path on this file
while not (src / 'src').exists() and src != src.parent:
    src = src.parent

sys.path.insert(0, str(src))
os.chdir(src)

In [ ]:
DATA_DIR = Path('data/intel')
MODEL_DIR = Path('models/cnn')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

### 1. Load Dataset

In [ ]:
CLASSES = []

for iterdir in TRAIN_DIR.iterdir():
    if iterdir.is_dir():
        CLASSES.append(iterdir.name)

CLASSES.sort()
print('Classes are ', CLASSES)

# 80:20 Split untuk Train & Test dataset
# Define Variables
IMG_SIZE = 150
BATCH_SIZE = 64
EPOCHS = 20
NUM_CLASSES = 6

# 1. Train
train_split_ = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, 
                                                           image_size = (IMG_SIZE, IMG_SIZE),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           seed = 42,
                                                           class_names = CLASSES,
                                                           validation_split  = 0.2,
                                                           subset = 'training',
                                                           )

# 2. Validation
validate_split_ = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,
                                                              image_size = (IMG_SIZE, IMG_SIZE),
                                                              batch_size = BATCH_SIZE,
                                                              label_mode = 'int',
                                                              seed = 42,
                                                              class_names = CLASSES,
                                                              validation_split = 0.2,
                                                              subset = 'validation',
                                                              )

# 3. Test
test_split_ = tf.keras.utils.image_dataset_from_directory(TEST_DIR,
                                                           image_size = (IMG_SIZE, IMG_SIZE),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           shuffle = False,
                                                           class_names = CLASSES,
                                                           )

norm = tf.keras.layers.Rescaling(1./255)
train_ds = train_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds = validate_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

label_list = []
for gambar, label in test_ds:
    label_numpy = label.numpy()

    label_list.append(label_numpy)

y_test = np.concatenate(label_list)
print(f'Train: {len(train_ds)}')
print(f'Validation: {len(val_ds)}')
print(f'Test: {len(y_test)}')

### 2. CNN

In [ ]:
GRID = {
    'n_conv':  [2, 4],
    'filters': [[32, 64], [64, 128]],
    'k_size': [3, 5],
    'pool': ['max', 'avg']
}

configs = list(itertools.product(GRID['n_conv'],
                                 GRID['filters'],
                                 GRID['k_size'],
                                 GRID['pool']
                                 ))

for i, (n, fltrs, krnl, pl) in enumerate(configs):
    print(f' [{i}] n_conv = {n}, filters={fltrs}, k_size={krnl}, pool={pl}')

In [ ]:
def build_cnn(n_conv, filters, k_size, pool):
    if pool == 'max':
        PoolLayer = tf.keras.layers.MaxPooling2D
    else:
        PoolLayer = tf.keras.layers.AveragePooling2D

    name_model = f'cnn_c{n_conv}_k{k_size}_{pool}'
    model = tf.keras.Sequential(name=name_model)
    model.add(tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)))

    for i in range (n_conv):
        fltr = filters[i % len(filters)]
        model.add(tf.keras.layers.Conv2D(fltr, k_size, activation='relu', padding='same'))
        model.add(PoolLayer(2, 2))

    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

temp = build_cnn(2, [32,64], 3, 'max')
temp.summary()

In [ ]:
HIST_PATH = MODEL_DIR / 'histories.json'
history = {}

if HIST_PATH.exists():
    with open(HIST_PATH) as filepath:
        history = json.load(filepath)

result_list = []
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

for i, (n_conv, fltrs, krnl, pl) in enumerate(configs):
    tag = f'c{n_conv}_f{"_".join(map(str,fltrs))}_k{krnl}_{pl}'

    save_path = MODEL_DIR / f'arch_{i:02d}_{tag}.h5'
    print(f'[{i+1}/16] {tag}', end=' ... ')

    if save_path.exists():
        model = tf.keras.models.load_model(save_path)
        print('model loaded (1)')

    else:
        model = build_cnn(n_conv, fltrs, krnl, pl)
        hist = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=[early_stop], verbose=0)
        
        model.save(save_path)
        history[tag] = hist.history

        with open(HIST_PATH, 'w') as filepath:
            json.dump(history, filepath)
        print(f'trained {len(hist.history["loss"])} epochs')

    y_pred = model.predict(test_ds, verbose=0).argmax(1)
    f1 = f1_score(y_test, y_pred, average='macro')

    result_list.append({
        'idx': i, 'tag': tag,
        'n_conv': n_conv, 'filters': str(fltrs),
        'k_size': krnl, 'pool': pl,
        'f1_macro': round(f1, 4),
        'params': model.count_params(),
    })
    print(f'  - F1: {f1:.4f}')

with open(MODEL_DIR / 'results.json', 'w') as fp:
    json.dump(result_list, fp, indent=2)
print('\nDone!')  


### 3. Hasil Training

In [ ]:
df = pd.DataFrame(result_list).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df[['tag','n_conv','filters','k_size','pool','f1_macro','params']])

best_row = df.iloc[0]
print(f'\nArsitektur terbaik: {best_row["tag"]} (F1={best_row["f1_macro"]:.4f})')

In [ ]:
# Rata-rata F1 per nilai hyperparameter
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
hparams   = [('n_conv','n Conv Layer'), ('filters','n Filter'),
             ('k_size','size Kernel'), ('pool','type Pooling')]

for ax, (col, title) in zip(axes.flat, hparams):
    grp = df.groupby(col)['f1_macro'].mean().sort_values(ascending=False)

    ax.bar(grp.index.astype(str), grp.values, color='steelblue')
    ax.set_title(title)
    ax.set_ylabel('Mean F1-macro')
    ax.set_ylim(max(0, (grp.min()-0.05)), (grp.max()+0.05))

    for j, (x, v) in enumerate(zip(grp.index.astype(str), grp.values)):
        ax.text(j, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Pengaruh Hyperparameter terhadap Macro F1', y=1.02)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'hyperparam_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
HIST_PATH = MODEL_DIR / 'histories.json'
if HIST_PATH.exists():
    with open(HIST_PATH) as fp:
        history.update(json.load(fp))

best_tag = best_row['tag']
if best_tag in history:
    hist = history[best_tag]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(hist['loss'], label='train')
    ax1.plot(hist['val_loss'], label='val')
    ax1.set_title('Loss')
    ax1.legend()

    ax2.plot(hist['accuracy'], label='train')
    ax2.plot(hist['val_accuracy'], label='val')
    ax2.set_title('Accuracy')
    ax2.legend()

    plt.suptitle(f'Training Curves - Arsitektur Terbaik ({best_tag})')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'best_curves.png', bbox_inches='tight')
    plt.show()

else:
    print("Tidak memiliki history")


if history:

    hparam_groups = {
        'n_conv':  {'label': 'Jumlah Conv Layer', 'vals': sorted({result['n_conv'] for result in result_list})},
        'k_size':  {'label': 'Ukuran Filter', 'vals': sorted({result['k_size'] for result in result_list})},
        'pool':    {'label': 'Jenis Pooling', 'vals': sorted({result['pool'] for result in result_list})},
        'filters': {'label': 'Filter per Layer', 'vals': sorted({result['filters'] for result in result_list})},
    }

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for param, info in hparam_groups.items():
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        for ci, val in enumerate(info['vals']):
            subset = []
            for result in result_list:
                if str(result[param]) == str(val):
                    subset.append(result)

            all_vl = []
            all_tl = []
            for s in subset:
                if s['tag'] in history:
                    vl = history[s['tag']]['val_loss']
                    tl = history[s['tag']]['loss']

                    all_vl.append(vl)
                    all_tl.append(tl)

            if not all_vl:
                continue
            
            max_e = 0
            for vl in all_vl:
                len_vl = len(vl)

                if (len_vl > max_e):
                    max_e = len_vl
                    
            pad = lambda arr: [v + [v[-1]] * (max_e - len(v)) for v in arr]
            mean_vl = np.array(pad(all_vl)).mean(axis=0)
            mean_tl = np.array(pad(all_tl)).mean(axis=0)

            c = colors[ci % len(colors)]
            ax1.plot(mean_tl, label=f'{param}={val}', color=c)
            ax2.plot(mean_vl, label=f'{param}={val}', color=c)

        ax1.set_title(f'Train Loss - {info["label"]}'); ax1.legend(); ax1.set_xlabel('Epoch')
        ax2.set_title(f'Val Loss - {info["label"]}');   ax2.legend(); ax2.set_xlabel('Epoch')
        plt.suptitle(f'Pengaruh {info["label"]} terhadap Loss')

        plt.tight_layout()
        plt.savefig(MODEL_DIR / f'loss_curves_{param}.png', bbox_inches='tight')
        plt.show()

else:
    print('Tidak ada history tersimpan.')


### 4. Keras v. From-Scratch

In [2]:
from src.cnn.model import CNNFromScratch

best_path  = MODEL_DIR / f'arch_{best_row["idx"]:02d}_{best_tag}.h5'
keras_model = tf.keras.models.load_model(best_path)

X_test_np = np.concatenate([x.numpy() for x, _ in test_ds])
print(f'X_test shape: {X_test_np.shape}')

# 1. Keras
time_start = time.time()

y_keras = keras_model.predict(test_ds, verbose=0).argmax(1)
t_keras = time.time() - time_start
f1_keras = f1_score(y_test, y_keras, average='macro')


# 2. Scracth
scratch = CNNFromScratch.from_keras(keras_model)
time_start = time.time()

y_scratch = scratch.predict_classes(X_test_np)
t_scratch = time.time() - time_start
f1_scratch = f1_score(y_test, y_scratch, average='macro')

print(f'\n{"":20s} {"F1-macro":>10s} {"Waktu (s)":>10s}')
print(f'{"Keras":20s} {f1_keras:>10.4f} {t_keras:>10.2f}')
print(f'{"From-Scratch":20s} {f1_scratch:>10.4f} {t_scratch:>10.2f}')
print(f'\nPrediksi identik: {np.all(y_keras == y_scratch)}')


ModuleNotFoundError: No module named 'src'

In [ ]:
print('=== Keras ===')
print(classification_report(y_test, y_keras, target_names=CLASSES))
print('\n=== From-Scratch ===')
print(classification_report(y_test, y_scratch, target_names=CLASSES))

### 5. Conv2D v. LocallyConnected2D

In [ ]:
IMG_SIZE_LC2D = 32

# 1. Train
train_split_LC2D = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, 
                                                           image_size = (IMG_SIZE_LC2D, IMG_SIZE_LC2D),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           seed = 42,
                                                           class_names = CLASSES,
                                                           )

# 2. Test
test_split_LC2D = tf.keras.utils.image_dataset_from_directory(TEST_DIR,
                                                           image_size = (IMG_SIZE_LC2D, IMG_SIZE_LC2D),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           shuffle = False,
                                                           class_names = CLASSES,
                                                           )

train_ds_LC2D = train_split_LC2D.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds_LC2D  = test_split_LC2D.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

label_list_LC2D = []
for gambar, label in test_ds_LC2D:
    label_numpy = label.numpy()

    label_list_LC2D.append(label_numpy)

y_test_LC2D= np.concatenate(label_list_LC2D)

In [ ]:
def build_LC2D(n_conv, filters, k_size, pool, img_size=IMG_SIZE_LC2D):
    if pool == 'max':
        PoolLayer = tf.keras.layers.MaxPooling2D
    else:
        PoolLayer = tf.keras.layers.AveragePooling2D
    
    name_model = f'lc2D_c{n_conv}_k{k_size}_{pool}'
    model = tf.keras.Sequential(name=name_model)
    model.add(tf.keras.layers.Input(shape=(img_size, img_size, 3))) # image size can be customized later

    for i in range(n_conv):
        fltr = filters[i % len(filters)]

        model.add(tf.keras.layers.LocallyConnected2D(fltr, k_size, activation='relu'))
        model.add(PoolLayer(2, 2))
    
    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

NameError: name 'IMG_SIZE_LC2D' is not defined

In [ ]:
# hyperparamters akan sama dengan best di Conv2D
n_conv_best = best_row['n_conv']
filters_best = eval(best_row['filters'])
k_size_best = best_row['k_size']
pool_best = best_row['pool']

lc2D_model = build_LC2D(n_conv_best, filters_best, k_size_best, pool_best)
lc2D_model.summary()


conv2D_32x32 = build_cnn(n_conv_best, filters_best, k_size_best, pool_best)

# change input size to 32x32 biar sama dengan LC2D
conv2D_32x32 = tf.keras.Sequential([tf.keras.layers.Input(shape=(IMG_SIZE_LC2D, IMG_SIZE_LC2D, 3))] +
                                [layer for layer in conv2D_32x32.layers[1:]],
                                name='conv2D_32x32')

conv2D_32x32.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
lc2d_save   = MODEL_DIR / f'lc2d_best.h5'
conv2D_32x32_save  = MODEL_DIR / f'conv2d_32x32_best.h5'

for model, path, dataset in [(conv2D_32x32, conv2D_32x32_save, train_ds_LC2D),
                             (lc2D_model, lc2d_save, train_ds_LC2D),
                             ]:
    
    name = model.name

    if path.exists():
        print(f'{name} is loaded')
        model.set_weights(tf.keras.models.load_model(path).get_weights())

    else:
        print(f'Training model: {name}')
        hist = model.fit(dataset, epochs = EPOCHS, validation_data = test_ds_LC2D, callbacks=[early_stop], verbose=0)
        model.save(path)
        history[name] = hist.history

        print(f' Done, went through ({len(hist.history["loss"])} epochs)')



In [ ]:
if lc2d_save.exists():
    lc2D_model = tf.keras.models.load_model(lc2d_save)
if conv2D_32x32_save.exists():
    conv2D_32x32 = tf.keras.models.load_model(conv2D_32x32_save)

y_conv2D_32x32 = conv2D_32x32.predict(test_ds_LC2D, verbose=0).argmax(1)
y_LC2D = lc2D_model.predict(test_ds_LC2D, verbose=0).argmax(1)

f1_conv2D_32x32 = f1_score(y_test_LC2D, y_conv2D_32x32, average='macro')
f1_LC2D    = f1_score(y_test_LC2D, y_LC2D,     average='macro')

print(f'\n{"":25s} {"F1-macro":>10s} {"Params":>12s}')
print(f'{"Conv2D (32x32)":25s} {f1_conv2D_32x32:>10.4f} {conv2D_32x32.count_params():>12,}')
print(f'{"LC2D   (32x32)":25s} {f1_LC2D:>10.4f} {lc2D_model.count_params():>12,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, hist in [('conv2d_32x32', history.get('conv2D_32x32')),
                   ('lc2d', history.get(lc2D_model.name))
                   ]:
    
    if hist is None:
        continue

    axes[0].plot(hist['loss'],      label=f'{name} train')
    axes[0].plot(hist['val_loss'],  label=f'{name} val', linestyle='--')
    axes[1].plot(hist['accuracy'],     label=f'{name} train')
    axes[1].plot(hist['val_accuracy'], label=f'{name} val', linestyle='--')

axes[0].set_title('Loss'); axes[0].legend(fontsize=8)
axes[1].set_title('Accuracy'); axes[1].legend(fontsize=8)
plt.suptitle('Conv2D vs LocallyConnected2D')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'conv_vs_lc2d.png', bbox_inches='tight')
plt.show()